# Unified ASR LoRA Fine-Tuning — switch model + dataset, everything else stays the same

One notebook, one registry of adapters for all four model families (OmniASR, FastConformer-CTC,
Qwen3-ASR, CohereAsr Transcribe). **Change `MODEL_NAME` and `DATASET_DIR` in Cell 2 — nothing
else needs to change.**

Each model family still needs its own pre-built venv/kernel (real dependency conflict: OmniASR's
`fairseq2n` pins `transformers==4.57.6`/`numpy<2`; Qwen/Cohere need `transformers>=5.13`/`numpy2`;
these cannot coexist in one interpreter). Restart this notebook's kernel to the one matching your
`MODEL_NAME` — Cell 2 prints which one and errors loudly if you picked the wrong one. All four
adapter classes are always *defined* here regardless of kernel (their real imports are lazy,
inside `load_base()`), so the notebook itself never needs to change — only which kernel you launch
it with, driven purely by which `MODEL_NAME` you picked.

| `MODEL_NAME` | kernel | venv |
|---|---|---|
| `omnilingual-asr/omniASR_LLM_300M`, `..._1B` | `omni_gpu` | `/workspace/venv_omni_gpu` |
| `nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0`, `..._pc_v1.0` | `nemo_gpu` | `/workspace/venv_nemo_gpu` |
| `Qwen/Qwen3-ASR-0.6B-hf` | `qwen_gpu` | `/workspace/venv_qwen_gpu` |
| `CohereLabs/cohere-transcribe-arabic-07-2026` (gated) | `qwen_gpu` | `/workspace/venv_qwen_gpu` |


In [1]:
# Cell 1 — Environment
# No single pip line here on purpose -- see the kernel table above. Whichever venv this
# kernel points at already has that model family's stack installed; the other three
# families' packages are imported lazily inside their adapter's load_base(), so an
# unused adapter class being *defined* here never requires its packages to be present.

import os, json, gc, math, time, random, hashlib, shutil, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import Any, Dict, List, Optional, Callable

import numpy as np, torch, torch.nn as nn
warnings.filterwarnings("ignore")
print(torch.__version__, torch.cuda.is_available())

2.8.0+cu128 True


## Cell 2 — The only two things you change

In [2]:
# ================================================================
# THE ONLY TWO THINGS YOU CHANGE
# ================================================================
MODEL_NAME  = "Qwen/Qwen3-ASR-0.6B-hf"
DATASET_DIR = "/workspace/asr/Palestinian-ASR/omnilingual_selected/apc_north_levantine_all_splits"
# ================================================================

KERNEL_BY_MODEL = {
    "omnilingual-asr/omniASR_LLM_1B":                    "omni_gpu (/workspace/venv_omni_gpu)",
    "omnilingual-asr/omniASR_LLM_300M":                  "omni_gpu (/workspace/venv_omni_gpu)",
    "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": "nemo_gpu (/workspace/venv_nemo_gpu)",
    "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0":  "nemo_gpu (/workspace/venv_nemo_gpu)",
    "Qwen/Qwen3-ASR-0.6B-hf":                            "qwen_gpu (/workspace/venv_qwen_gpu)",
    "CohereLabs/cohere-transcribe-arabic-07-2026":       "qwen_gpu (/workspace/venv_qwen_gpu)",
}
# Each adapter's own native language-conditioning code (OmniASR script codes vs. plain ISO
# codes elsewhere) -- kept out of the two knobs above since it's determined by MODEL_NAME,
# not something you'd want to pick independently.
LANG_BY_MODEL = {
    "omnilingual-asr/omniASR_LLM_1B":                    "arb_Arab",
    "omnilingual-asr/omniASR_LLM_300M":                  "arb_Arab",
    "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": "ar",
    "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0":  "ar",
    "Qwen/Qwen3-ASR-0.6B-hf":                            "ar",
    "CohereLabs/cohere-transcribe-arabic-07-2026":       "ar",
}
LANG = LANG_BY_MODEL[MODEL_NAME]

SMOKE_TEST    = True                # tiny subsets + 2 epochs
SEED          = 42

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ROOT          = Path(os.environ.get("ASR_ENV_ROOT", "/workspace/asr_env"))
MODEL_CACHE   = ROOT / "models"
PRED_DIR      = ROOT / "preds"
METRIC_DIR    = ROOT / "metrics"
CKPT_DIR      = ROOT / "checkpoints"
for d in (MODEL_CACHE, PRED_DIR, METRIC_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
import mlflow
# mlflow 3.x deprecated the plain "file:" tracking backend (raises unless
# MLFLOW_ALLOW_FILE_STORE=true) -- sqlite is the currently-recommended local backend and
# also lets `mlflow ui --backend-store-uri ...` browse runs later. Artifact location is
# set explicitly on first creation since sqlite backends don't default one on their own.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
_MLFLOW_EXPERIMENT = "arabic-asr-unified"
if mlflow.get_experiment_by_name(_MLFLOW_EXPERIMENT) is None:
    mlflow.create_experiment(_MLFLOW_EXPERIMENT, artifact_location=f"file:{ROOT / 'mlruns' / _MLFLOW_EXPERIMENT}")
mlflow.set_experiment(_MLFLOW_EXPERIMENT)

if MODEL_NAME.startswith("CohereLabs/"):
    # The repo is GATED: verify credentials up front so the failure mode is obvious.
    from huggingface_hub import get_token
    _tok = get_token() or os.environ.get("HF_TOKEN")
    if not _tok:
        print("[WARN] No Hugging Face token found (hf auth login / HF_TOKEN). "
              f"{MODEL_NAME} is a gated repo -- the model download will fail with 401 "
              "until a token whose account accepted the license is available.")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print(f"MODEL_NAME={MODEL_NAME}")
print(f"kernel should be: {KERNEL_BY_MODEL.get(MODEL_NAME, '?? not in KERNEL_BY_MODEL')}")
print(f"DEVICE={DEVICE} | dtype={COMPUTE_DTYPE} | ROOT={ROOT} | DATASET_DIR={DATASET_DIR}")

MODEL_NAME=Qwen/Qwen3-ASR-0.6B-hf
kernel should be: qwen_gpu (/workspace/venv_qwen_gpu)
DEVICE=cuda | dtype=torch.bfloat16 | ROOT=/workspace/asr_env | DATASET_DIR=/workspace/asr/Palestinian-ASR/omnilingual_selected/apc_north_levantine_all_splits


## Cell 3 — Arabic normalization + WER/CER

In [3]:
import re, unicodedata, jiwer

_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
_PUNC = re.compile(r"[^\w\s\u0621-\u064A]")

def normalize_ar(t: str) -> str:
    """Diacritic strip, tatweel removal, alef/ya/ta-marbuta unification."""
    if t is None: return ""
    t = unicodedata.normalize("NFKC", str(t))
    t = _DIAC.sub("", t)
    t = re.sub("[\u0622\u0623\u0625\u0671]", "\u0627", t)   # alef variants -> alef
    t = t.replace("\u0649", "\u064A")                          # alef maqsura -> ya
    t = t.replace("\u0629", "\u0647")                          # ta marbuta -> ha
    t = t.replace("\u0624", "\u0648").replace("\u0626", "\u064A")
    t = _PUNC.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def compute_wer_cer(preds, refs, normalize=True):
    if normalize:
        preds = [normalize_ar(p) for p in preds]
        refs  = [normalize_ar(r) for r in refs]
    keep = [(p, r) for p, r in zip(preds, refs) if r.strip()]
    if not keep: return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    p, r = zip(*keep)
    return {"wer": jiwer.wer(list(r), list(p)),
            "cer": jiwer.cer(list(r), list(p)),
            "n": len(r)}

## Cell 4 — ConfigAPI

Per-model LoRA + training defaults. Your Whisper study settled on `r=32, alpha=32, lr=1e-4, AdamW 8-bit, patience=3` — here patience is 4 per spec and the OmniASR target modules follow the wav2vec2_llama attention naming.

In [4]:
@dataclass
class LoRAConfigSpec:
    r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    target_modules: Optional[List[str]] = None
    modules_to_save: Optional[List[str]] = None
    task_type: Optional[str] = None

@dataclass
class TrainConfigSpec:
    num_epochs: int = 50
    early_stopping_patience: int = 3
    metric_for_best: str = "wer"
    greater_is_better: bool = False
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    optim: str = "adamw_bnb_8bit"
    bf16: bool = True
    gradient_checkpointing: bool = False
    dataloader_num_workers: int = 0
    max_audio_seconds: float = 30.0
    max_label_tokens: int = 256
    save_total_limit: int = 3
    save_steps: Optional[int] = None      # None -> auto: max(200, steps_per_epoch // 3)
    dataloader_pin_memory: bool = True
    dataloader_persistent_workers: bool = True
    dataloader_prefetch_factor: int = 4

class ConfigAPI:
    """Single source of truth for per-model hyperparameters, across all four families.

    NOTE on target_modules per family (real API differences, not guesses -- see each
    adapter class docstring below for the verification evidence):
      * OmniASR (fairseq2, not transformers): StandardMultiheadAttention
        (q_proj/k_proj/v_proj/output_proj) + GLUFeedForwardNetwork (gate_proj/inner_proj).
      * FastConformer-CTC (NeMo): linear_q/linear_k/linear_v/linear_out (attention) +
        linear1/linear2 (feed-forward).
      * Qwen3-ASR (transformers-native): standard q/k/v/o_proj + gate/up/down_proj.
      * CohereAsr (transformers-native, gated): target_modules=None -> discovered at
        runtime in CohereAsrAdapter.apply_lora() via named_modules() introspection.
    """
    _LORA = {
        "omnilingual-asr/omniASR_LLM_1B": LoRAConfigSpec(
            r=32, lora_alpha=32,
            target_modules=["q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"]),
        "omnilingual-asr/omniASR_LLM_300M": LoRAConfigSpec(
            r=32, lora_alpha=32,
            target_modules=["q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"]),
        "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": LoRAConfigSpec(
            target_modules=["linear_q","linear_k","linear_v","linear_out","linear1","linear2"]),
        "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0": LoRAConfigSpec(
            target_modules=["linear_q","linear_k","linear_v","linear_out","linear1","linear2"]),
        "Qwen/Qwen3-ASR-0.6B-hf": LoRAConfigSpec(
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
        "CohereLabs/cohere-transcribe-arabic-07-2026": LoRAConfigSpec(target_modules=None),
    }
    _TRAIN = {
        "omnilingual-asr/omniASR_LLM_1B": TrainConfigSpec(
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            gradient_checkpointing=True, dataloader_num_workers=4,
            # 40s = OmniASR's HARD inference ceiling (see the dedicated notebook's Cell 4
            # comment for the full verified-on-GPU rationale).
            max_audio_seconds=40.0),
        "omnilingual-asr/omniASR_LLM_300M": TrainConfigSpec(
            per_device_train_batch_size=4, gradient_accumulation_steps=4,
            gradient_checkpointing=True, dataloader_num_workers=4,
            max_audio_seconds=40.0),
        "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": TrainConfigSpec(
            per_device_train_batch_size=2, gradient_accumulation_steps=4),
        "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0": TrainConfigSpec(
            per_device_train_batch_size=2, gradient_accumulation_steps=4),
        "Qwen/Qwen3-ASR-0.6B-hf": TrainConfigSpec(
            per_device_train_batch_size=2, gradient_accumulation_steps=4),
        "CohereLabs/cohere-transcribe-arabic-07-2026": TrainConfigSpec(
            per_device_train_batch_size=1, gradient_accumulation_steps=8),   # 2B model
    }
    @classmethod
    def lora(cls, name)  -> LoRAConfigSpec:  return cls._LORA.get(name, LoRAConfigSpec())
    @classmethod
    def train(cls, name) -> TrainConfigSpec: return cls._TRAIN.get(name, TrainConfigSpec())

print(json.dumps(asdict(ConfigAPI.lora(MODEL_NAME)), indent=2))
print(json.dumps(asdict(ConfigAPI.train(MODEL_NAME)), indent=2))

{
  "r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "bias": "none",
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "modules_to_save": null,
  "task_type": null
}
{
  "num_epochs": 50,
  "early_stopping_patience": 3,
  "metric_for_best": "wer",
  "greater_is_better": false,
  "per_device_train_batch_size": 2,
  "per_device_eval_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.0,
  "max_grad_norm": 1.0,
  "optim": "adamw_bnb_8bit",
  "bf16": true,
  "gradient_checkpointing": false,
  "dataloader_num_workers": 0,
  "max_audio_seconds": 30.0,
  "max_label_tokens": 256,
  "save_total_limit": 3,
  "save_steps": null,
  "dataloader_pin_memory": true,
  "dataloader_persistent_workers": true,
  "dataloader_prefetch_factor": 4
}


## Cell 5 — ModelAdapter ABC

The whole point of the abstraction. Every model-specific difference lives behind one of these six methods:

| method | Whisper | Qwen3-ASR | CTC (NemoCTC) | OmniASR LLM |
|---|---|---|---|---|
| `preprocess` | log-mel + tokenized text | **chat template** wrap | text -> char ids, no BOS/EOS | audio tensor + `lang` token prefix |
| `collate` | pad mel + `-100` label pad | pad ids + attn mask | pad audio + label lens | pad waveform + pad label ids |
| `loss_type` | seq2seq CE | seq2seq CE | **CTC** | seq2seq CE |

In [5]:
from abc import ABC, abstractmethod

class ModelAdapter(ABC):
    name: str
    loss_type: str = "seq2seq"          # or "ctc"
    supports_unsloth: bool = False

    def __init__(self, model_name: str, lang: str = LANG):
        self.model_name = model_name; self.lang = lang
        self.model = None; self.processor = None

    @abstractmethod
    def load_base(self): ...
    @abstractmethod
    def preprocess(self, example: Dict) -> Dict: ...
    @abstractmethod
    def collate(self, features: List[Dict]) -> Dict[str, Any]: ...
    @abstractmethod
    def generate(self, batch: Dict) -> List[str]: ...

    def apply_lora(self, spec: "LoRAConfigSpec"):
        """Default: standard PEFT LoRA wrapping self.model in place -- correct whenever the
        target projections are real torch.nn.Linear (Qwen3-ASR's case). Adapters whose LoRA
        can't go through this path override it: OmniASRAdapter (manual injection -- fairseq2
        Linear isn't a torch.nn.Linear subclass), ConformerCTCAdapter (keeps the PEFT wrapper
        in self.peft, not self.model, so the raw NeMo API stays reachable on self.model),
        CohereAsrAdapter (target_modules is discovered at runtime first)."""
        if self.supports_unsloth:
            try:
                from unsloth import FastModel
                self.model = FastModel.get_peft_model(
                    self.model, r=spec.r, lora_alpha=spec.lora_alpha,
                    lora_dropout=spec.lora_dropout, bias=spec.bias,
                    target_modules=spec.target_modules, use_gradient_checkpointing="unsloth")
                print("[lora] unsloth"); return self.model
            except Exception as e:
                print(f"[lora] unsloth unavailable ({e}); falling back to PEFT")
        from peft import LoraConfig, get_peft_model
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        self.model = get_peft_model(self.model, LoraConfig(**kw))
        self.model.print_trainable_parameters()
        print("[lora] peft"); return self.model

    def train_step(self, batch) -> torch.Tensor:
        out = self.model(**batch)
        return out.loss if hasattr(out, "loss") else out["loss"]

    def trainable_parameters(self) -> List["torch.nn.Parameter"]:
        return [p for p in self.model.parameters() if p.requires_grad]

    def save_checkpoint(self, dir_):
        """Default: self.model IS the PEFT-wrapped module. Concrete adapters whose LoRA
        lives elsewhere (self.peft, manual injection) override this."""
        Path(dir_).mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(str(dir_))

    def load_checkpoint(self, dir_):
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        sd = load_file(str(Path(dir_) / "adapter_model.safetensors"))
        set_peft_model_state_dict(self.model, sd)

## Cell 6 — OmniASR adapter (shared by 1B and 300M)

One class, two entries in the registry. The only delta is the `model_card` string, exactly as you predicted.

In [6]:
import math

class _LoRALinear(nn.Module):
    """Manual LoRA wrapper. OmniASR's projections are `fairseq2.nn.projection.Linear`, which is
    NOT a `torch.nn.Linear` subclass, so neither PEFT nor Unsloth can wrap them. We inject a
    low-rank side path ourselves: y = base(x) + scaling * dropout(x) @ A^T @ B^T, B zero-init so
    the initial delta is 0. Works on any module exposing a 2-D `.weight`."""
    def __init__(self, base: nn.Module, r: int, alpha: int, dropout: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad_(False)
        out_f, in_f = base.weight.shape
        dt, dev = base.weight.dtype, base.weight.device
        self.lora_A = nn.Parameter(torch.zeros(r, in_f, dtype=dt, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r, dtype=dt, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        out = self.base(x)
        delta = self.drop(x) @ self.lora_A.t() @ self.lora_B.t()
        return out + self.scaling * delta


class OmniASRAdapter(ModelAdapter):
    """fairseq2 wav2vec2_llama. Verified against omnilingual-asr@main source AND a real CPU run
    of omniASR_LLM_300M (1.63B params). See DISCOVERY.md / SMOKE_RESULTS.md.

      * pipeline.model -> Wav2Vec2LlamaModel;  pipeline.tokenizer -> Tokenizer
      * tokenizer.create_encoder() takes NO lang; decode via create_decoder(skip_special_tokens=True)
      * TRAINING: loss = model(Seq2SeqBatch(...)); model builds `audio [lang] <bos> text <eos>` and
        masks the loss internally. Pad text with pad_idx (NOT -100). seq_lens must be list[int].
      * INFERENCE: pipeline.transcribe(list[dict{waveform,sample_rate}], lang=[...], batch_size=n)
      * LoRA: projections are fairseq2.nn.projection.Linear (NOT torch.nn.Linear) -> PEFT/Unsloth
        can't wrap them, so we inject LoRA manually via _LoRALinear.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    CARD = {"omnilingual-asr/omniASR_LLM_1B":   "omniASR_LLM_1B",
            "omnilingual-asr/omniASR_LLM_300M": "omniASR_LLM_300M"}

    def __init__(self, model_name, lang=LANG):
        super().__init__(model_name, lang)
        self.card = self.CARD[model_name]
        self.pipeline = None; self.tokenizer = None
        self.pad_idx = 0
        self._encoder = None; self._decoder = None
        self.derived_target_modules = None

    def load_base(self):
        local = MODEL_CACHE / self.card
        from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
        if local.exists() and any(local.iterdir()):
            print(f"[load] local -> {local}")
        else:
            print(f"[load] downloading {self.card} -> {local}")
            local.mkdir(parents=True, exist_ok=True)
        # FAIRSEQ2_CACHE_DIR controls the checkpoint cache (verified on a real 300M load).
        os.environ.setdefault("FAIRSEQ2_CACHE_DIR", str(local))

        self.pipeline  = ASRInferencePipeline(self.card, device=DEVICE)
        self.model     = self.pipeline.model
        self.tokenizer = self.pipeline.tokenizer
        self._encoder  = self.tokenizer.create_encoder()
        self._decoder  = self.tokenizer.create_decoder(skip_special_tokens=True)
        self.pad_idx   = getattr(self.tokenizer.vocab_info, "pad_idx", 0) or 0

        from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
        assert self.lang in supported_langs, f"{self.lang} not in supported_langs"

        self.derived_target_modules = self._derive_target_modules()
        print(f"[load] pad_idx={self.pad_idx} | derived target_modules={self.derived_target_modules}")
        return self.model

    def _derive_target_modules(self):
        """Detect projection leaf names by DUCK TYPING (2-D `.weight`), because fairseq2's Linear
        is not a torch.nn.Linear subclass and isinstance(mod, nn.Linear) would miss all of them."""
        want = {"q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"}
        found = set()
        for name, mod in self.model.named_modules():
            leaf = name.split(".")[-1]
            w = getattr(mod, "weight", None)
            if leaf in want and w is not None and getattr(w, "ndim", 0) == 2:
                found.add(leaf)
        return sorted(found) or sorted(want)

    def apply_lora(self, spec: LoRAConfigSpec):
        """Manual LoRA injection (PEFT/Unsloth can't wrap fairseq2.nn.projection.Linear)."""
        targets = set(self.derived_target_modules or spec.target_modules)
        replaced = 0
        for mod_name, mod in list(self.model.named_modules()):
            leaf = mod_name.split(".")[-1]
            w = getattr(mod, "weight", None)
            if (leaf in targets and w is not None and getattr(w, "ndim", 0) == 2
                    and not isinstance(mod, _LoRALinear)):
                parent = self.model.get_submodule(mod_name.rsplit(".", 1)[0]) if "." in mod_name else self.model
                setattr(parent, leaf, _LoRALinear(mod, spec.r, spec.lora_alpha, spec.lora_dropout))
                replaced += 1
        for n, p in self.model.named_parameters():
            p.requires_grad_("lora_" in n)
        n_tr = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[lora] manual injection into {replaced} fairseq2 Linear layers | trainable params={n_tr}")
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa; audio = librosa.resample(np.asarray(audio, dtype=np.float32),
                                                     orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        ids  = self._encode(text)
        return {"input_values": np.asarray(audio, dtype=np.float32),
                "labels": ids, "text": text,
                "audio_len": len(audio) / 16000.0}

    def _encode(self, text):
        if self._encoder is None: return []
        return self._encoder(text).tolist()

    def _decode(self, ids):
        if self._decoder is None: return ""
        import torch as _t
        return str(self._decoder(_t.as_tensor(ids, dtype=_t.int64)))

    def collate(self, feats):
        maxa = max(len(f["input_values"]) for f in feats)
        maxl = max(len(f["labels"]) for f in feats) or 1
        wav  = torch.zeros(len(feats), maxa, dtype=torch.float32)
        mask = torch.zeros(len(feats), maxa, dtype=torch.long)
        lab  = torch.full((len(feats), maxl), self.pad_idx, dtype=torch.long)
        lab_lens = torch.zeros(len(feats), dtype=torch.long)
        for i, f in enumerate(feats):
            a = torch.as_tensor(f["input_values"], dtype=torch.float32)
            wav[i, :len(a)] = a; mask[i, :len(a)] = 1
            if len(f["labels"]):
                lab[i, :len(f["labels"])] = torch.as_tensor(f["labels"], dtype=torch.long)
                lab_lens[i] = len(f["labels"])
        return {"input_values": wav, "attention_mask": mask, "labels": lab,
                "label_lengths": lab_lens,
                "lang": [self.lang]*len(feats), "text": [f["text"] for f in feats]}

    def train_step(self, batch) -> torch.Tensor:
        from fairseq2.datasets.batch import Seq2SeqBatch
        wav  = batch["input_values"]; mask = batch["attention_mask"]; lab = batch["labels"]
        # fairseq2 Seq2SeqBatch requires seq_lens as list[int], NOT tensors.
        src_lens = mask.sum(dim=1).to(torch.long).tolist()
        if "label_lengths" in batch:
            tgt_lens = batch["label_lengths"].to(torch.long).tolist()
        else:
            tgt_lens = (lab != self.pad_idx).sum(dim=1).to(torch.long).tolist()
        model_dtype = next(self.model.parameters()).dtype
        dev = getattr(self.model, "device", DEVICE)
        langs = batch.get("lang", [self.lang]*wav.shape[0])
        seq2seq = Seq2SeqBatch(
            source_seqs     = wav.to(dev, model_dtype),
            source_seq_lens = src_lens,
            target_seqs     = lab.to(dev).to(torch.long),
            target_seq_lens = tgt_lens,
            example         = {"lang": list(langs)},
        )
        out = self.model(seq2seq)
        return out if torch.is_tensor(out) else out[0]

    @torch.no_grad()
    def generate(self, batch):
        wav  = batch["input_values"]; mask = batch["attention_mask"]
        inp = []
        for i in range(wav.shape[0]):
            n = int(mask[i].sum().item())
            inp.append({"waveform": wav[i, :n].float().cpu().numpy(), "sample_rate": 16000})
        langs = batch.get("lang", [self.lang]*len(inp))
        out = self.pipeline.transcribe(inp, lang=list(langs), batch_size=len(inp))
        return [str(o) for o in out]

    def save_checkpoint(self, dir_):
        """Manual LoRA (no PEFT wrapper for fairseq2 Linear) -> save just the LoRA tensors."""
        dir_ = Path(dir_); dir_.mkdir(parents=True, exist_ok=True)
        sd = {k: v for k, v in self.model.state_dict().items() if "lora_" in k}
        torch.save(sd, dir_ / "adapter.pt")

    def load_checkpoint(self, dir_):
        sd = torch.load(Path(dir_) / "adapter.pt", map_location=DEVICE)
        self.model.load_state_dict(sd, strict=False)


## FastConformer-CTC adapter

In [7]:
class ConformerCTCAdapter(ModelAdapter):
    '''nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0 in pure-CTC mode.
       Verified on GPU (2026-07-18):
         * load:  ASRModel.from_pretrained -> EncDecHybridRNNTCTCBPEModel (114.6M),
                  change_decoding_strategy(decoder_type="ctc")
         * TRAIN: enc,enc_len = model(input_signal, input_signal_length)
                  log_probs   = model.ctc_decoder(encoder_output=enc)
                  loss        = model.ctc_loss(log_probs, targets, enc_len, target_lengths)
         * INFER: model.transcribe([wav_paths]) -> [Hypothesis.text]   (CTC greedy)
         * LoRA:  real PEFT on conformer attention+FF linears, injected in place. The PEFT
                  wrapper is kept in self.peft (not self.model) so the raw NeMo API stays
                  reachable on self.model for forward/ctc/transcribe.
    '''
    loss_type = "ctc"
    supports_unsloth = False

    def __init__(self, model_name, lang=LANG):
        super().__init__(model_name, lang)
        self.peft = None

    def load_base(self):
        import nemo.collections.asr as nemo_asr
        print(f"[load] {self.model_name}")
        self.model = nemo_asr.models.ASRModel.from_pretrained(self.model_name, map_location=DEVICE)
        self.model.change_decoding_strategy(decoder_type="ctc")   # drive the CTC head
        self.peft = None
        return self.model

    def apply_lora(self, spec: "LoRAConfigSpec"):
        from peft import LoraConfig, get_peft_model
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        # get_peft_model injects lora.Linear into self.model's submodules IN PLACE and returns a
        # PeftModel wrapper. We keep self.model (the NeMo object) for forward/ctc/transcribe and
        # use self.peft only to save/print/reload the adapter.
        self.peft = get_peft_model(self.model, LoraConfig(**kw))
        self.peft.print_trainable_parameters()
        print("[lora] peft"); return self.peft

    def trainable_parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def save_checkpoint(self, dir_):
        Path(dir_).mkdir(parents=True, exist_ok=True)
        self.peft.save_pretrained(str(dir_))

    def load_checkpoint(self, dir_):
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        sd = load_file(str(Path(dir_) / "adapter_model.safetensors"))
        set_peft_model_state_dict(self.peft, sd)

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def collate(self, feats):
        sigs = [torch.from_numpy(np.asarray(f["audio"], dtype=np.float32)) for f in feats]
        lens = torch.tensor([int(s.numel()) for s in sigs], dtype=torch.long)
        maxT = int(lens.max())
        sig  = torch.zeros(len(sigs), maxT, dtype=torch.float32)
        for i, s in enumerate(sigs): sig[i, :s.numel()] = s
        toks = [self.model.tokenizer.text_to_ids(f["text"]) for f in feats]
        tlen = torch.tensor([len(t) for t in toks], dtype=torch.long)
        maxL = max(1, int(tlen.max()))
        tgt  = torch.zeros(len(toks), maxL, dtype=torch.long)
        for i, t in enumerate(toks):
            if t: tgt[i, :len(t)] = torch.tensor(t, dtype=torch.long)
        return {"input_signal": sig, "input_signal_length": lens,
                "targets": tgt, "target_lengths": tlen,
                "text": [f["text"] for f in feats], "_audio": [f["audio"] for f in feats]}

    def train_step(self, batch) -> torch.Tensor:
        enc, enc_len = self.model(input_signal=batch["input_signal"].to(DEVICE),
                                  input_signal_length=batch["input_signal_length"].to(DEVICE))
        log_probs = self.model.ctc_decoder(encoder_output=enc)
        return self.model.ctc_loss(log_probs=log_probs,
                                   targets=batch["targets"].to(DEVICE),
                                   input_lengths=enc_len,
                                   target_lengths=batch["target_lengths"].to(DEVICE))

    @torch.no_grad()
    def generate(self, batch):
        import tempfile, soundfile as sf
        was_training = self.model.training
        self.model.eval()
        tmp = tempfile.mkdtemp(prefix="ctc_infer_"); paths = []
        for i, a in enumerate(batch["_audio"]):
            p = os.path.join(tmp, f"{i}.wav")
            sf.write(p, np.asarray(a, dtype=np.float32), 16000); paths.append(p)
        hyps = self.model.transcribe(paths, batch_size=len(paths), verbose=False)
        out = [(h.text if hasattr(h, "text") else str(h)) for h in hyps]
        for p in paths:
            try: os.unlink(p)
            except Exception: pass
        try: os.rmdir(tmp)
        except Exception: pass
        if was_training: self.model.train()
        return out

## Qwen3-ASR adapter

In [8]:
class Qwen3ASRAdapter(ModelAdapter):
    """Qwen/Qwen3-ASR-0.6B-hf (transformers-native). Verified on GPU (2026-07-16):
      * load:   AutoProcessor + Qwen3ASRForConditionalGeneration (bf16)
      * TRAIN:  processor.apply_chat_template(chat, output_labels=True) -> loss = model(**in).loss
                chat = user turn holding {type:text, text:transcript} + {type:audio, audio:ndarray}
      * INFER:  processor.apply_transcription_request(audio=[...], language=[...]) -> model.generate
                -> processor.decode(gen, return_format="transcription_only")
      * inputs MUST be cast to the model dtype (BatchFeature.to(device, dtype) casts float only).
      * LoRA:   real PEFT on q/k/v/o_proj + gate/up/down_proj -- uses ModelAdapter's default
                apply_lora/save_checkpoint/load_checkpoint as-is, no override needed.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    def load_base(self):
        from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration
        print(f"[load] {self.model_name}")
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = Qwen3ASRForConditionalGeneration.from_pretrained(
            self.model_name, dtype=COMPUTE_DTYPE).to(DEVICE)
        self.model.config.use_cache = False
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def collate(self, feats):
        chat = [[{"role": "user", "content": [
                    {"type": "text",  "text":  f["text"]},
                    {"type": "audio", "audio": f["audio"]}]}]
                for f in feats]
        inputs = self.processor.apply_chat_template(
            chat, tokenize=True, return_dict=True, output_labels=True)
        batch = dict(inputs)                       # BatchFeature -> plain dict of tensors
        batch["text"]   = [f["text"] for f in feats]
        batch["_audio"] = [f["audio"] for f in feats]
        return batch

    _MODEL_KEYS = ("input_ids", "attention_mask", "input_features", "input_features_mask", "labels")

    def train_step(self, batch) -> torch.Tensor:
        inputs = {}
        for k in self._MODEL_KEYS:
            v = batch.get(k)
            if v is None: continue
            v = v.to(DEVICE)
            if v.is_floating_point(): v = v.to(COMPUTE_DTYPE)   # input_features -> bf16
            inputs[k] = v
        return self.model(**inputs).loss

    @torch.no_grad()
    def generate(self, batch):
        audios = list(batch["_audio"])
        req = self.processor.apply_transcription_request(
            audio=audios, language=[self.lang] * len(audios))
        req = req.to(DEVICE, COMPUTE_DTYPE)
        out_ids = self.model.generate(**req, max_new_tokens=256)
        gen = out_ids[:, req["input_ids"].shape[1]:]
        return [str(t) for t in self.processor.decode(gen, return_format="transcription_only")]

## CohereAsr adapter

In [9]:
class CohereAsrAdapter(ModelAdapter):
    """CohereLabs/cohere-transcribe-arabic-07-2026 (transformers-native CohereAsr).
      * load:   AutoProcessor + CohereAsrForConditionalGeneration (bf16)  [GATED repo]
      * TRAIN:  explicit teacher forcing -> loss = model(**in).loss
      * INFER:  processor(audio, language, sampling_rate=16000) -> model.generate
                -> strip prompt -> tokenizer.batch_decode(skip_special_tokens=True)
                (+ processor._reassemble_chunk_texts when clips got chunked)
      * inputs MUST be cast to the model dtype (BatchFeature.to(device, dtype) casts float only).
      * LoRA:   real PEFT on discovered encoder/decoder nn.Linear projections -- target_modules
                is None in ConfigAPI for this model, discovered here at runtime.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    def load_base(self):
        from transformers import AutoProcessor, CohereAsrForConditionalGeneration
        print(f"[load] {self.model_name}")
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = CohereAsrForConditionalGeneration.from_pretrained(
            self.model_name, dtype=COMPUTE_DTYPE).to(DEVICE)
        self.model.config.use_cache = False
        tok = self.processor.tokenizer
        self._prompt_ids = list(self.processor.get_decoder_prompt_ids(
            language=self.lang, punctuation=True))
        eos = self.model.generation_config.eos_token_id
        if isinstance(eos, (list, tuple)): eos = eos[0]
        self._eos_id = int(eos if eos is not None else tok.eos_token_id)
        pad = self.model.config.pad_token_id
        if pad is None: pad = tok.pad_token_id if tok.pad_token_id is not None else self._eos_id
        self._pad_id = int(pad)
        print(f"[load] prompt={tok.convert_ids_to_tokens(self._prompt_ids)} "
              f"eos={self._eos_id} pad={self._pad_id}")
        return self.model

    def _discover_lora_targets(self) -> List[str]:
        """Leaf names of nn.Linear modules that look like attention/FF projections."""
        # Attention CONTENT projections + feed-forward only -- matching the FastConformer
        # sibling adapter. Deliberately excluded: `relative_k_proj` (projects positional
        # embeddings, not content), bare `linear` (the conv-subsampling frontend), and the
        # output heads.
        pat = re.compile(r"^(q|k|v|o)_proj$|^(gate|up|down)_proj$"
                         r"|^linear_(q|k|v|out)$|^linear[12]$|^fc[12]$|^w[123]$")
        names = {n.split(".")[-1] for n, m in self.model.named_modules() if isinstance(m, nn.Linear)}
        targets = sorted(n for n in names if pat.match(n) and n not in {"lm_head", "proj_out"})
        if not targets:
            raise RuntimeError(f"No LoRA targets matched. Available Linear leaves: {sorted(names)}")
        return targets

    def apply_lora(self, spec: "LoRAConfigSpec"):
        """PEFT LoRA. CohereAsr projections are torch.nn.Linear, so this Just Works -- only
        the target_modules discovery step differs from the base class default."""
        from peft import LoraConfig, get_peft_model
        if spec.target_modules is None:
            spec = replace(spec, target_modules=self._discover_lora_targets())
            print(f"[lora] discovered target_modules: {spec.target_modules}")
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        self.model = get_peft_model(self.model, LoraConfig(**kw))
        self.model.print_trainable_parameters()
        print("[lora] peft"); return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def _features(self, audios):
        enc = self.processor(audio=[np.asarray(a, dtype=np.float32) for a in audios],
                             language=self.lang, sampling_rate=16000)
        chunk_index = enc.pop("audio_chunk_index", None)
        return enc, chunk_index

    def collate(self, feats):
        enc, chunk_index = self._features([f["audio"] for f in feats])
        if enc["input_features"].shape[0] != len(feats):
            raise RuntimeError(
                f"feature extractor chunked {len(feats)} clips into "
                f"{enc['input_features'].shape[0]} rows -- keep training clips <= max_audio_clip_s")
        tok = self.processor.tokenizer
        P = self._prompt_ids
        seqs = [P + tok(f["text"], add_special_tokens=False)["input_ids"] + [self._eos_id]
                for f in feats]
        maxT = max(len(s) for s in seqs) - 1
        dec_in  = torch.full((len(seqs), maxT), self._pad_id, dtype=torch.long)
        labels  = torch.full((len(seqs), maxT), -100, dtype=torch.long)
        dmask   = torch.zeros((len(seqs), maxT), dtype=torch.long)
        for i, s in enumerate(seqs):
            L = len(s) - 1
            dec_in[i, :L] = torch.tensor(s[:-1], dtype=torch.long)
            dmask[i, :L] = 1
            lab = torch.tensor(s[1:], dtype=torch.long)
            lab[:len(P) - 1] = -100                      # don't train on the prompt
            labels[i, :L] = lab
        batch = {"input_features": enc["input_features"],
                 "decoder_input_ids": dec_in, "decoder_attention_mask": dmask,
                 "labels": labels}
        if "attention_mask" in enc: batch["attention_mask"] = enc["attention_mask"]
        batch["text"]   = [f["text"] for f in feats]
        batch["_audio"] = [f["audio"] for f in feats]
        return batch

    _MODEL_KEYS = ("input_features", "attention_mask", "decoder_input_ids",
                   "decoder_attention_mask", "labels")

    def train_step(self, batch) -> torch.Tensor:
        inputs = {}
        for k in self._MODEL_KEYS:
            v = batch.get(k)
            if v is None: continue
            v = v.to(DEVICE)
            if v.is_floating_point(): v = v.to(COMPUTE_DTYPE)   # input_features -> bf16
            inputs[k] = v
        return self.model(**inputs).loss

    @torch.no_grad()
    def generate(self, batch):
        enc, chunk_index = self._features(list(batch["_audio"]))
        enc = enc.to(DEVICE)
        req = {k: (v.to(COMPUTE_DTYPE) if torch.is_tensor(v) and v.is_floating_point() else v)
               for k, v in enc.items()}
        out_ids = self.model.generate(**req, max_new_tokens=256)
        gen = out_ids[:, req["decoder_input_ids"].shape[1]:]
        texts = self.processor.tokenizer.batch_decode(gen, skip_special_tokens=True)
        if chunk_index is not None and any(c[1] is not None for c in chunk_index):
            texts = self.processor._reassemble_chunk_texts(texts, chunk_index, " ")
        return [str(t).strip() for t in texts]

## Cell 7 — Registry (all four families)

In [10]:
REGISTRY: Dict[str, Callable[..., ModelAdapter]] = {
    "omnilingual-asr/omniASR_LLM_1B":                    OmniASRAdapter,
    "omnilingual-asr/omniASR_LLM_300M":                  OmniASRAdapter,
    "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": ConformerCTCAdapter,
    "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0":  ConformerCTCAdapter,   # no-diacritics twin
    "Qwen/Qwen3-ASR-0.6B-hf":                            Qwen3ASRAdapter,
    "CohereLabs/cohere-transcribe-arabic-07-2026":       CohereAsrAdapter,
}

def get_adapter(name, **kw) -> ModelAdapter:
    if name not in REGISTRY: raise KeyError(f"{name} not registered. Have: {list(REGISTRY)}")
    a = REGISTRY[name](name, **kw); a.name = name; return a

## Cell 8 — Datasets

Assumed ready. Swap the loader for your QASR/MGB2 splits.

In [11]:
from datasets import load_from_disk, Audio, Dataset
import soundfile as sf, io, random as _random

# Real North-Levantine (Palestinian) Arabic data (see HANDOFF.md item 6) instead of the
# Common Voice placeholder. Single flat Dataset (not DatasetDict) — audio is FLAC bytes.
# Decode via soundfile, NOT the datasets library's built-in Audio(decode=True) path: this
# `datasets` version hard-requires `torchcodec` for that (both decode AND re-encode, so even
# .map() on an Audio(decode=False) column fails), and pulling torchcodec risks clobbering the
# carefully-pinned cu128 torch build. Materializing into a fresh Dataset.from_generator sidesteps
# the Audio feature type entirely — the new "audio" column is inferred as a plain struct.
REAL_DATA_DIR = Path(DATASET_DIR)  # set in Cell 2

def _materialize_audio(ds):
    def gen():
        for row in ds:
            wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            out = dict(row)
            out["audio"] = {"array": wav, "sampling_rate": sr}
            yield out
    return Dataset.from_generator(gen)

def load_splits(smoke=SMOKE_TEST):
    ds = load_from_disk(str(REAL_DATA_DIR))
    if "raw_text" in ds.column_names and "text" not in ds.column_names:
        ds = ds.rename_column("raw_text", "text")
    ds = ds.cast_column("audio", Audio(decode=False))
    if smoke:
        # Most rows in this corpus are full multi-sentence prompts (median ~70s). We now allow
        # up to TrainConfigSpec.max_audio_seconds=40 (OmniASR's true inference ceiling; the old
        # 30s cap was arbitrary -- training on >30s is verified working). To actually EXERCISE
        # the raised limit end-to-end, the smoke test deliberately picks clips from the newly-
        # allowed (30, 40]s band so both train (fwd/bwd) and generate (transcribe, <=40s ok)
        # run on a >30s sample. Filtering via the plain "duration" column triggers no audio
        # decode. (>40s clips still train but crash transcribe -- hence the 40s upper bound.)
        # Generalized to whichever model is selected in Cell 2 (each family's own dedicated
        # notebook may carve out a tighter/model-specific band -- e.g. OmniASR's own notebook
        # deliberately exercises its raised (30,40]s ceiling -- this unified notebook uses the
        # simple, general "at or under this model's own limit" rule instead).
        max_s = ConfigAPI.train(MODEL_NAME).max_audio_seconds
        durations = ds["duration"]
        short_idx = [i for i, d in enumerate(durations) if d <= max_s]
        _random.Random(SEED).shuffle(short_idx)
        # 1 fake train / 1 fake val / 1 fake test sample — real audio+text, arbitrary
        # disjoint single-row split assignment (not the corpus's own split column).
        splits = {"train": ds.select(short_idx[0:1]),
                  "validation": ds.select(short_idx[1:2]),
                  "test": ds.select(short_idx[2:3])}
    else:
        ds = ds.shuffle(seed=SEED)
        n = len(ds)
        n_tr, n_va = int(n * 0.8), int(n * 0.9)
        splits = {"train": ds.select(range(0, n_tr)),
                  "validation": ds.select(range(n_tr, n_va)),
                  "test": ds.select(range(n_va, n))}
    for k in splits: splits[k] = _materialize_audio(splits[k])
    return splits

SPLITS = load_splits()
{k: len(v) for k, v in SPLITS.items()}

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 1 examples [00:00,  9.24 examples/s]

Generating train split: 1 examples [00:00,  6.39 examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 1 examples [00:00, 10.97 examples/s]

{'train': 1, 'validation': 1, 'test': 1}

## Cell 9 — PredictAPI (cached)

Cache key = `model + split + dataset fingerprint + stage`. Re-running the cell loads from disk instead of re-decoding.

In [12]:
def _fingerprint(ds) -> str:
    try: h = ds._fingerprint
    except Exception: h = str(len(ds))
    return hashlib.md5(f"{h}{len(ds)}".encode()).hexdigest()[:10]

class PredictAPI:
    @staticmethod
    def _path(model_name, split, ds, stage):
        slug = model_name.replace("/", "__")
        return PRED_DIR / f"{slug}__{split}__{_fingerprint(ds)}__{stage}.json"

    @staticmethod
    def run(adapter, ds, split="test", stage="base", batch_size=4, force=False):
        p = PredictAPI._path(adapter.name, split, ds, stage)
        if p.exists() and not force:
            print(f"[predict] CACHE HIT -> {p.name}")
            return json.loads(p.read_text(encoding="utf-8"))

        print(f"[predict] generating ({stage}, {split}, n={len(ds)})")
        feats = [adapter.preprocess(ex) for ex in ds]
        preds, refs = [], []
        adapter.model.eval()
        for i in range(0, len(feats), batch_size):
            b = adapter.collate(feats[i:i+batch_size])
            b_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            preds.extend(adapter.generate(b_dev)); refs.extend(b["text"])
            print(f"  {min(i+batch_size,len(feats))}/{len(feats)}", end="\r")
        rec = {"model": adapter.name, "split": split, "stage": stage,
               "predictions": preds, "references": refs,
               "n": len(preds), "ts": time.time()}
        p.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n[predict] saved -> {p.name}")
        return rec

## Cell 10 — EvaluateAPI (cached)

In [13]:
class EvaluateAPI:
    @staticmethod
    def _path(model_name, split, stage, pred_record=None):
        slug = model_name.replace('/', '__')
        if pred_record is None:
            return METRIC_DIR / f"{slug}__{split}__{stage}.json"
        # Content-address the metric to the exact predictions it scores. PredictAPI already
        # keys on the dataset fingerprint; without the same discipline here a metric computed
        # on an older corpus gets silently re-served after the data changes, producing a wrong
        # base WER and a meaningless base->tuned delta.
        payload = json.dumps([pred_record["predictions"], pred_record["references"]],
                             ensure_ascii=False, sort_keys=True).encode("utf-8")
        h = hashlib.md5(payload).hexdigest()[:10]
        return METRIC_DIR / f"{slug}__{split}__{h}__{stage}.json"

    @staticmethod
    def run(model_name, pred_record, split="test", stage="base", force=False):
        p = EvaluateAPI._path(model_name, split, stage, pred_record)
        if p.exists() and not force:
            m = json.loads(p.read_text()); print(f"[eval] CACHE HIT -> {m}"); return m
        m = compute_wer_cer(pred_record["predictions"], pred_record["references"])
        m.update({"model": model_name, "split": split, "stage": stage})
        p.write_text(json.dumps(m, indent=2))
        print(f"[eval] WER={m['wer']:.4f} CER={m['cer']:.4f} (n={m['n']}) -> {p.name}")
        return m

## Cell 11 — Build adapter + load base model

In [14]:
set_seed()
adapter = get_adapter(MODEL_NAME, lang=LANG)
adapter.load_base()
n_params = sum(p.numel() for p in adapter.model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M params | loss_type={adapter.loss_type} | unsloth={adapter.supports_unsloth}")

[load] Qwen/Qwen3-ASR-0.6B-hf


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

Loading weights:  12%|█▏        | 73/611 [00:00<00:00, 728.98it/s]

Loading weights:  24%|██▍       | 146/611 [00:00<00:01, 458.43it/s]

Loading weights:  34%|███▍      | 208/611 [00:00<00:00, 511.23it/s]

...

Loading weights:  86%|████████▌ | 523/611 [00:01<00:00, 265.74it/s]

Loading weights:  92%|█████████▏| 563/611 [00:01<00:00, 292.76it/s]

Loading weights:  97%|█████████▋| 594/611 [00:01<00:00, 273.59it/s]

Loading weights: 100%|██████████| 611/611 [00:01<00:00, 356.77it/s]

Qwen/Qwen3-ASR-0.6B-hf: 782.4M params | loss_type=seq2seq | unsloth=False


## Cell 12 — Baseline preds + eval on test (cached)

In [15]:
base_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="base",
                              batch_size=ConfigAPI.train(MODEL_NAME).per_device_eval_batch_size)
base_metrics = EvaluateAPI.run(MODEL_NAME, base_preds, split="test", stage="base")

for p, r in list(zip(base_preds["predictions"], base_preds["references"]))[:3]:
    print(f"REF : {r}\nHYP : {p}\n")

[predict] generating (base, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__38c34837ce__base.json
[eval] CACHE HIT -> {'wer': 0.43902439024390244, 'cer': 0.13452914798206278, 'n': 1, 'model': 'Qwen/Qwen3-ASR-0.6B-hf', 'split': 'test', 'stage': 'base'}
REF : بعدين منحط البهارات فوين وبعد ما منحط البهارات فوين منخلين يغلو ليستو شوي بعدين منشلح فوين الفريكه وبس تستوي مناكلا ولا اطيب من هيك بتاخد مده الاستوا noise لالا تاريبا حوالي شي ساعه علي الغاز مجرد ما تستوي بتصير جاهزه للاكل
HYP : بعدين منحط البهارات فوقه وبعد ما منحط البهارات فوقه من خليه يغلو ليستوي شوي. بعدين منشل حفوقه لفريقي وبس تستوي مني كله ولا أطيب من هيك. بتاخذ مدة الاستوى لأهلا تقريبا حوالي شه ساعة على الغاز مجرد ما تستوي بصير جاهز للاكل.



## Cell 13 — Apply LoRA

In [16]:
lora_spec  = ConfigAPI.lora(MODEL_NAME)
train_spec = ConfigAPI.train(MODEL_NAME)
if SMOKE_TEST:
    train_spec.num_epochs = 2
    train_spec.early_stopping_patience = 4

adapter.apply_lora(lora_spec)

trainable params: 23,281,664 || all params: 805,707,776 || trainable%: 2.8896
[lora] peft


PeftModel(
  (base_model): LoraModel(
    (model): Qwen3ASRForConditionalGeneration(
      (model): Qwen3ASRModel(
        (audio_tower): Qwen3ASREncoder(
          (positional_embedding): SinusoidsPositionEmbedding()
          (layers): ModuleList(
            (0-17): 18 x Qwen3ASRAudioEncoderLayer(
              (self_attn): Qwen3ASRAudioAttention(
                (k_proj): lora.Linear(
                  (base_layer): Linear(in_features=896, out_features=896, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=896, out_features=32, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=32, out_features=896, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()

## Cell 14 — TrainAPI

Custom loop rather than `Seq2SeqTrainer`: the fairseq2 module isn't a `PreTrainedModel`, so the HF Trainer's save/load/generate hooks don't apply. Per-epoch logging of train loss, val loss, val WER, val CER; early stopping on WER with patience 4; best-WER checkpoint only.

In [17]:
import mlflow
from contextlib import nullcontext
from torch.utils.data import DataLoader, Sampler

def _amp(spec):
    """bf16 autocast on CUDA; no-op elsewhere so the loop also runs on CPU."""
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

class _ListDS(torch.utils.data.Dataset):
    def __init__(self, feats): self.f = feats
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i]

class LengthGroupedSampler(Sampler):
    """Buckets examples into batch_size-sized windows sorted by audio length (for padding
    efficiency), then shuffles the *order of windows* every epoch -- so training still sees
    a different batch order each pass instead of a frozen short->long sweep. A fresh
    torch.randperm before the sort also jitters which items land in which window across
    epochs (tie-breaking), not just the window order."""
    def __init__(self, lengths, batch_size):
        self.lengths = lengths; self.batch_size = batch_size
    def __len__(self): return len(self.lengths)
    def __iter__(self):
        idx = torch.randperm(len(self.lengths)).tolist()
        idx.sort(key=lambda i: self.lengths[i])
        buckets = [idx[i:i + self.batch_size] for i in range(0, len(idx), self.batch_size)]
        order = torch.randperm(len(buckets)).tolist()
        out = []
        for b in order: out.extend(buckets[b])
        return iter(out)

def _rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])

class TrainAPI:
    @staticmethod
    def _prep(adapter, ds, spec):
        """Pre-flight: preprocess + drop over-long / empty items before the loop."""
        feats, dropped = [], 0
        for ex in ds:
            f = adapter.preprocess(ex)
            if f["audio_len"] > spec.max_audio_seconds: dropped += 1; continue
            if not f["text"].strip():                   dropped += 1; continue
            if len(f.get("labels") or []) > spec.max_label_tokens:
                f["labels"] = f["labels"][:spec.max_label_tokens]
            feats.append(f)
        print(f"[prep] kept {len(feats)}, dropped {dropped}")
        return feats

    @staticmethod
    @torch.no_grad()
    def _validate(adapter, loader, spec):
        adapter.model.eval(); losses, preds, refs = [], [], []
        for b in loader:
            g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            try:
                with _amp(spec):
                    losses.append(float(adapter.train_step(g)))
            except Exception as e:
                print(f"[val] loss skipped: {e}")
            preds.extend(adapter.generate(g)); refs.extend(b["text"])
        m = compute_wer_cer(preds, refs)
        m["val_loss"] = float(np.mean(losses)) if losses else float("nan")
        m["_preds"], m["_refs"] = preds, refs
        return m

    @staticmethod
    def run(adapter, splits, spec: TrainConfigSpec, lora_spec: LoRAConfigSpec, resume: bool = True):
        slug = adapter.name.replace("/", "__")
        run_root = CKPT_DIR / slug
        best_dir = run_root / "best"; best_dir.mkdir(parents=True, exist_ok=True)

        tr_f = TrainAPI._prep(adapter, splits["train"], spec)
        va_f = TrainAPI._prep(adapter, splits["validation"], spec)
        tr_kw = {}
        if spec.dataloader_num_workers > 0:
            tr_kw["persistent_workers"] = spec.dataloader_persistent_workers
            tr_kw["prefetch_factor"] = spec.dataloader_prefetch_factor
        tr = DataLoader(_ListDS(tr_f), batch_size=spec.per_device_train_batch_size,
                        sampler=LengthGroupedSampler([f["audio_len"] for f in tr_f],
                                                     spec.per_device_train_batch_size),
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers,
                        pin_memory=(spec.dataloader_pin_memory and DEVICE == "cuda"),
                        drop_last=False, **tr_kw)
        va = DataLoader(_ListDS(va_f), batch_size=spec.per_device_eval_batch_size, shuffle=False,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers)

        params = adapter.trainable_parameters()
        opt = None
        if DEVICE == "cuda":            # bitsandbytes 8-bit optimizers are CUDA-only
            try:
                import bitsandbytes as bnb
                opt = bnb.optim.AdamW8bit(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)
            except Exception as e:
                print(f"[opt] AdamW8bit unavailable ({e}); using torch.AdamW")
        if opt is None:
            opt = torch.optim.AdamW(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)

        steps_pe = max(1, math.ceil(len(tr) / spec.gradient_accumulation_steps))
        total    = steps_pe * spec.num_epochs
        save_steps = spec.save_steps or max(200, steps_pe // 3)
        from transformers import get_linear_schedule_with_warmup
        sched = get_linear_schedule_with_warmup(opt, int(total * spec.warmup_ratio), total)

        best_wer, bad_epochs, gstep, history, start_epoch, mlflow_run_id = \
            float("inf"), 0, 0, [], 1, None

        ckpts = sorted(run_root.glob("ckpt_step*"))
        if resume and ckpts:
            last = ckpts[-1]
            try:
                state = json.loads((last / "trainer_state.json").read_text())
                adapter.load_checkpoint(last)
                # weights_only=False: these are our own trusted local checkpoint files, not
                # untrusted downloads. Needed because torch >=2.6 defaults weights_only=True,
                # which rejects the numpy-backed RNG state (numpy.random.get_state() pickles via
                # numpy's own _reconstruct, not in the default safe-globals allowlist) and can
                # also reject optimizer state depending on the optimizer's internals.
                opt.load_state_dict(torch.load(last / "optimizer.pt", map_location=DEVICE, weights_only=False))
                sched.load_state_dict(torch.load(last / "scheduler.pt", map_location=DEVICE, weights_only=False))
                _restore_rng(torch.load(last / "rng.pt", map_location="cpu", weights_only=False))
                best_wer, bad_epochs = state["best_wer"], state["bad_epochs"]
                gstep, start_epoch = state["gstep"], state["epoch"] + 1
                history, mlflow_run_id = state["history"], state.get("mlflow_run_id")
                print(f"[resume] epoch {start_epoch} gstep {gstep} best_wer {best_wer:.4f} <- {last}")
            except Exception as e:
                print(f"[resume] failed ({e}); starting fresh")

        try:
            mlflow.start_run(run_id=mlflow_run_id, run_name=f"{slug}-lora")
        except Exception as e:
            # A resumed run_id can be unusable for reasons outside our control (belongs to
            # a different active experiment -- e.g. this same checkpoint dir was previously
            # used by a sibling notebook under a different MLflow experiment name; a
            # manually deleted run; a different MLFLOW_TRACKING_URI). Never let bookkeeping
            # block real training -- fall back to a fresh run instead of crashing.
            print(f"[mlflow] could not resume run {mlflow_run_id} ({e}); starting a new run")
            mlflow_run_id = None
            mlflow.start_run(run_id=None, run_name=f"{slug}-lora")
        mlflow_run_id = mlflow.active_run().info.run_id
        try:
            params_flat = {f"train.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                           for k, v in asdict(spec).items()}
            params_flat.update({f"lora.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                                for k, v in asdict(lora_spec).items()})
            params_flat.update({"model": adapter.name, "lang": LANG, "smoke": SMOKE_TEST,
                                "save_steps": save_steps})
            mlflow.log_params(params_flat)
        except Exception as e:
            print(f"[mlflow] log_params skipped ({e})")
        try:
            mlflow.set_tags({"dataset_version": REAL_DATA_DIR.name,
                             "hardware": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                             "stage": "dev" if SMOKE_TEST else "experiment"})
        except Exception as e:
            print(f"[mlflow] set_tags skipped ({e})")

        def _save_numbered_ckpt(epoch):
            nonlocal gstep
            d = run_root / f"ckpt_step{gstep:08d}"; d.mkdir(parents=True, exist_ok=True)
            adapter.save_checkpoint(d)
            torch.save(opt.state_dict(), d / "optimizer.pt")
            torch.save(sched.state_dict(), d / "scheduler.pt")
            torch.save(_rng_state(), d / "rng.pt")
            (d / "trainer_state.json").write_text(json.dumps(
                {"epoch": epoch, "gstep": gstep, "best_wer": best_wer, "bad_epochs": bad_epochs,
                 "history": history, "mlflow_run_id": mlflow_run_id}, indent=2))
            kept = sorted(run_root.glob("ckpt_step*"))
            for old in kept[:-spec.save_total_limit] if spec.save_total_limit > 0 else kept:
                shutil.rmtree(old, ignore_errors=True)
            return d

        for epoch in range(start_epoch, spec.num_epochs + 1):
            t_epoch = time.time()
            adapter.model.train(); ep_loss, nb = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, b in enumerate(tr):
                t_step = time.time()
                g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
                with _amp(spec):
                    loss = adapter.train_step(g) / spec.gradient_accumulation_steps
                if not torch.isfinite(loss):
                    print(f"[nan] step {i} non-finite loss, skipping batch")
                    opt.zero_grad(set_to_none=True); continue
                loss.backward()
                if (i + 1) % spec.gradient_accumulation_steps == 0 or (i + 1) == len(tr):
                    gnorm = torch.nn.utils.clip_grad_norm_(params, spec.max_grad_norm)
                    if not torch.isfinite(gnorm):
                        print(f"[nan] step {i} non-finite grad norm, skipping update")
                        opt.zero_grad(set_to_none=True); continue
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                    mem = torch.cuda.memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0
                    mlflow.log_metrics({"train/step_loss": float(loss) * spec.gradient_accumulation_steps,
                                        "train/grad_norm": float(gnorm),
                                        "train/lr": sched.get_last_lr()[0],
                                        "train/step_time_s": time.time() - t_step,
                                        "train/gpu_mem_gb": mem}, step=gstep)
                    if gstep % save_steps == 0:
                        _save_numbered_ckpt(epoch)
                ep_loss += float(loss) * spec.gradient_accumulation_steps; nb += 1

            train_loss = ep_loss / max(nb, 1)
            vm = TrainAPI._validate(adapter, va, spec)
            row = {"epoch": epoch, "train_loss": train_loss, "val_loss": vm["val_loss"],
                   "val_wer": vm["wer"], "val_cer": vm["cer"]}
            history.append(row)
            throughput = sum(f["audio_len"] for f in tr_f) / max(time.time() - t_epoch, 1e-6)
            mlflow.log_metrics({"epoch": epoch, "train/loss": train_loss, "val/loss": vm["val_loss"],
                                "val/wer": vm["wer"], "val/cer": vm["cer"],
                                "train/throughput_audio_s_per_s": throughput,
                                "train/epoch_time_s": time.time() - t_epoch}, step=gstep)
            try:
                import pandas as pd
                qual = pd.DataFrame({"reference": vm["_refs"][:5], "hypothesis": vm["_preds"][:5]})
                mlflow.log_table(qual, artifact_file=f"qualitative/epoch_{epoch:03d}.json")
            except Exception as e:
                print(f"[mlflow] qualitative table skipped ({e})")
            print(f"epoch {epoch:>3} | train {train_loss:.4f} | val {vm['val_loss']:.4f} "
                  f"| WER {vm['wer']:.4f} | CER {vm['cer']:.4f}")

            # ---- best-WER checkpoint (protected) + early stopping ----
            if vm["wer"] < best_wer - 1e-6:
                best_wer, bad_epochs = vm["wer"], 0
                adapter.save_checkpoint(best_dir)
                (best_dir / "best.json").write_text(json.dumps({**row, "gstep": gstep}, indent=2))
                print(f"  -> new best WER {best_wer:.4f}, saved to {best_dir}")
            else:
                bad_epochs += 1
                print(f"  -> no improvement ({bad_epochs}/{spec.early_stopping_patience})")

            # ---- epoch-boundary checkpoint, regardless of step count: clean resume fallback ----
            _save_numbered_ckpt(epoch)

            if bad_epochs >= spec.early_stopping_patience:
                print(f"[early-stop] epoch {epoch}, best WER {best_wer:.4f}"); break

        mlflow.log_metric("best_val_wer", best_wer, step=gstep)
        (run_root / "history.json").write_text(json.dumps(history, indent=2))
        return {"best_wer": best_wer, "best_dir": str(best_dir), "history": history,
                "mlflow_run_id": mlflow_run_id}


## Cell 15 — Train

In [18]:
set_seed()
train_out = TrainAPI.run(adapter, SPLITS, train_spec, lora_spec)
print(f"best val WER: {train_out['best_wer']:.4f} @ {train_out['best_dir']}")

[prep] kept 1, dropped 0


[prep] kept 1, dropped 0


[resume] epoch 3 gstep 2 best_wer 0.4651 <- /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/ckpt_step00000002
[mlflow] could not resume run abc07a45585b4ae2bc179fdcdbcabab2 (Cannot start run with ID abc07a45585b4ae2bc179fdcdbcabab2 because active experiment ID does not match environment run ID. Make sure --experiment-name or --experiment-id matches experiment set with set_experiment(), or just use command-line arguments); starting a new run


best val WER: 0.4651 @ /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best


## Cell 16 — Load best checkpoint, predict + evaluate on test, save

In [19]:
best = Path(train_out["best_dir"])
try:
    adapter.load_checkpoint(best)
    print(f"[ckpt] restored best adapter <- {best}")
except Exception as e:
    print(f"[ckpt] restore failed ({e})")

tuned_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="tuned",
                               batch_size=train_spec.per_device_eval_batch_size, force=True)
tuned_metrics = EvaluateAPI.run(MODEL_NAME, tuned_preds, split="test", stage="tuned", force=True)

dataset_hours  = {k: sum(v["duration"]) / 3600.0 for k, v in SPLITS.items()}
dataset_counts = {k: len(v) for k, v in SPLITS.items()}

summary = {
    "model": MODEL_NAME, "lang": LANG, "smoke_test": SMOKE_TEST,
    "base":  {"wer": base_metrics["wer"],  "cer": base_metrics["cer"]},
    "tuned": {"wer": tuned_metrics["wer"], "cer": tuned_metrics["cer"]},
    "delta": {"wer": base_metrics["wer"] - tuned_metrics["wer"],
              "cer": base_metrics["cer"] - tuned_metrics["cer"]},
    "best_val_wer": train_out["best_wer"],
    "dataset_counts": dataset_counts, "dataset_hours": dataset_hours,
    "lora": asdict(lora_spec), "train": asdict(train_spec),
}
sp = METRIC_DIR / f"{MODEL_NAME.replace('/','__')}__SUMMARY.json"
sp.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

mlflow.log_metrics({"test/base_wer": base_metrics["wer"],   "test/base_cer": base_metrics["cer"],
                    "test/tuned_wer": tuned_metrics["wer"], "test/tuned_cer": tuned_metrics["cer"],
                    **{f"data/{k}_hours": v for k, v in dataset_hours.items()},
                    **{f"data/{k}_count": v for k, v in dataset_counts.items()}})
try:
    mlflow.log_artifact(str(sp), artifact_path="summary")
    import pandas as pd
    mlflow.log_table(pd.DataFrame({"reference": tuned_preds["references"],
                                   "base_hyp": base_preds["predictions"],
                                   "tuned_hyp": tuned_preds["predictions"]}),
                     artifact_file="qualitative/test_predictions.json")
except Exception as e:
    print(f"[mlflow] artifact logging skipped ({e})")
mlflow.end_run()
print(json.dumps(summary, indent=2, ensure_ascii=False))


[ckpt] restored best adapter <- /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best
[predict] generating (tuned, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__38c34837ce__tuned.json
[eval] WER=0.4878 CER=0.1300 (n=1) -> Qwen__Qwen3-ASR-0.6B-hf__test__5a740bfa26__tuned.json


{
  "model": "Qwen/Qwen3-ASR-0.6B-hf",
  "lang": "ar",
  "smoke_test": true,
  "base": {
    "wer": 0.43902439024390244,
    "cer": 0.13452914798206278
  },
  "tuned": {
    "wer": 0.4878048780487805,
    "cer": 0.13004484304932734
  },
  "delta": {
    "wer": -0.04878048780487804,
    "cer": 0.004484304932735439
  },
  "best_val_wer": 0.46511627906976744,
  "dataset_counts": {
    "train": 1,
    "validation": 1,
    "test": 1
  },
  "dataset_hours": {
    "train": 0.008036799768518519,
    "validation": 0.0077747627314814815,
    "test": 0.006103252314814815
  },
  "lora": {
    "r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "modules_to_save": null,
    "task_type": null
  },
  "train": {
    "num_epochs": 2,
    "early_stopping_patience": 4,
    "metric_for_best": "wer",
    "greater_is_better": false,


## Cell 17 — One-shot smoke wrapper (currently-selected MODEL_NAME only)

Same code path regardless of which family is selected. Unlike the OmniASR-only dedicated notebook (which loops 300M/1B in one kernel since both share `fairseq2n`), this unified notebook can only exercise `MODEL_NAME` as currently set in Cell 2 -- the other three families need a different kernel entirely.

In [20]:
def smoke(model_name, splits):
    set_seed()
    a = get_adapter(model_name, lang=LANG); a.load_base()
    bp = PredictAPI.run(a, splits["test"], "test", "base", batch_size=2)
    bm = EvaluateAPI.run(model_name, bp, "test", "base")
    ls, ts = ConfigAPI.lora(model_name), ConfigAPI.train(model_name)
    ts.num_epochs = 2; ts.per_device_train_batch_size = 1; ts.gradient_accumulation_steps = 2
    a.apply_lora(ls)
    out = TrainAPI.run(a, splits, ts, ls)
    tp = PredictAPI.run(a, splits["test"], "test", "tuned", batch_size=2, force=True)
    tm = EvaluateAPI.run(model_name, tp, "test", "tuned", force=True)
    if mlflow.active_run() is not None:   # TrainAPI.run() started one; this loop never reaches
        mlflow.end_run()                  # the Cell 16 save-results cell that normally ends it
    del a.model, a; gc.collect(); torch.cuda.empty_cache()
    return {"model": model_name, "base_wer": bm["wer"], "tuned_wer": tm["wer"],
            "best_val_wer": out["best_wer"], "status": "PASS"}

# Only the currently-selected MODEL_NAME -- the whole point of the venv/kernel split in
# Cell 2 is that one kernel can only ever have one family's dependencies installed, so
# looping across families here (like each dedicated notebook loops across its own
# family's size variants) isn't possible in the unified notebook.
results = []
for m in [MODEL_NAME]:
    try:
        results.append(smoke(m, SPLITS))
    except Exception as e:
        import traceback; traceback.print_exc()
        results.append({"model": m, "status": f"FAIL: {e}"})
    print("=" * 70)

import pandas as pd
pd.DataFrame(results)

[load] Qwen/Qwen3-ASR-0.6B-hf


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

Loading weights:  65%|██████▍   | 395/611 [00:00<00:00, 3928.35it/s]

Loading weights: 100%|██████████| 611/611 [00:00<00:00, 3669.74it/s]

[predict] CACHE HIT -> Qwen__Qwen3-ASR-0.6B-hf__test__38c34837ce__base.json
[eval] CACHE HIT -> {'wer': 0.43902439024390244, 'cer': 0.13452914798206278, 'n': 1, 'model': 'Qwen/Qwen3-ASR-0.6B-hf', 'split': 'test', 'stage': 'base'}


...

[resume] epoch 3 gstep 2 best_wer 0.4651 <- /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/ckpt_step00000002
[mlflow] could not resume run abc07a45585b4ae2bc179fdcdbcabab2 (Cannot start run with ID abc07a45585b4ae2bc179fdcdbcabab2 because active experiment ID does not match environment run ID. Make sure --experiment-name or --experiment-id matches experiment set with set_experiment(), or just use command-line arguments); starting a new run


[predict] generating (tuned, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__38c34837ce__tuned.json
[eval] WER=0.4878 CER=0.1300 (n=1) -> Qwen__Qwen3-ASR-0.6B-hf__test__5a740bfa26__tuned.json


,model,base_wer,tuned_wer,best_val_wer,status
0,Qwen/Qwen3-ASR-0.6B-hf,0.439024,0.487805,0.465116,PASS
